In [4]:
import os
import numpy as np

In [5]:
DATASET = "archive"

Ignoring the metadata of the dataset since that is not necessary for the task at hand

In [43]:
lines = []

with open(os.path.join(DATASET, "movie_lines.txt"), "r") as content:
    content = content.readlines() # Read Line by Line
    # Postprocessing
    for line in content:
        processed = line.split("+++$+++")[-1]
        lines.append(processed)

Do the lines need to be coherent, or is it enough to train the dataset as it is ? My idea: Combine all the lines from each movie into one set; We can later break it down into context length size; I think basic coherence is needed. 

In [20]:
class Conversation:

    # Class Variables -- Unchanged
    LINES = "movie_lines.txt"
    METADATA = "movie_titles_metadata.txt"

    def __init__(self, root: str):

        self.root = root
        assert os.path.exists(os.path.join(root, self.LINES)) and os.path.exists(os.path.join(root, self.METADATA)), f"INVALID ROOT : {os.path.join(root, self.LINES), os.path.join(root, self.METADATA)} \n"

    
    def movie_map(self):
        """
        Function Mapping Movie -> Tag (m0, m1, .... m125, etc.)
        """

        self.movies = {} # {label : title}

        # Open txt file and create mapping
        with open(os.path.join(self.root, self.METADATA), "r") as content:
            # Read line by line, one movie at a time
            content = content.readlines()

            for data in content:
                # Process raw content to get label and title
                label, title = data.split("+++$+++")[0: 2]
                # Strip whitespace from label and title
                label, title = label.strip(), title.strip()
                # Verify if repear
                if not label in self.movies:
                    self.movies[label] = title
                # Throw error
                else: 
                    print("METADATA repeat! Terminate!")
                    return False
        return True

    def dialogue_map(self):
        """
        Group dialogues from movies
        """
        status = self.movie_map()
        
        if not status: return 0 # Terminate if error encountered

        self.dialogues = {} # {title : string}

        # Open line text file and group based on movie title
        with open(os.path.join(self.root, self.LINES), "r") as content:
            # Read line by line
            content = content.readlines()

            for line in content:
                # Process raw content to get movie and content
                tag, speaker, dialogue = line.split("+++$+++")[-3:]
                # Strip whitespaces
                tag, speaker, dialogue = tag.strip(), speaker.strip(), dialogue.strip()

                try:
                    # Get Title, map Title -> Dialogues
                    title = self.movies[tag]
                    
                    # Dict Addition
                    if title in self.dialogues:
                        self.dialogues[title] += f"{speaker} : {dialogue} \n"
                    
                    else:
                        self.dialogues[title] = f"{speaker} : {dialogue} \n"

                except Exception as e:

                    print(f"Invalid Line encountered : {line} \n")
                    print(f"ERROR: {e} \n")



In [22]:
dataset = Conversation(root = "archive")

dataset.dialogue_map() 

title = "10 things i hate about you"

# Testing one sample
print(f"Movie Title: {title} \n{dataset.dialogues[title]}")

Movie Title: 10 things i hate about you 
BIANCA : They do not! 
CAMERON : They do to! 
BIANCA : I hope so. 
CAMERON : She okay? 
BIANCA : Let's go. 
CAMERON : Wow 
BIANCA : Okay -- you're gonna need to learn how to lie. 
CAMERON : No 
BIANCA : I'm kidding.  You know how sometimes you just become this "persona"?  And you don't know how to quit? 
BIANCA : Like my fear of wearing pastels? 
CAMERON : The "real you". 
BIANCA : What good stuff? 
CAMERON : I figured you'd get to the good stuff eventually. 
CAMERON : Thank God!  If I had to hear one more story about your coiffure... 
BIANCA : Me.  This endless ...blonde babble. I'm like, boring myself. 
CAMERON : What crap? 
BIANCA : do you listen to this crap? 
CAMERON : No... 
BIANCA : Then Guillermo says, "If you go any lighter, you're gonna look like an extra on 90210." 
CAMERON : You always been this selfish? 
BIANCA : But 
CAMERON : Then that's all you had to say. 
BIANCA : Well, no... 
CAMERON : You never wanted to go out with 'me, did 